# FraudShield Colab Training Notebook

This notebook runs the **training-first FraudShield stack** directly from the repo.

It uses:
- `train.py` for Colab-friendly curriculum + QLoRA training
- `evaluate.py` for fixed-task evaluation
- `configs/colab_qlora_grpo.json` for reproducible settings

The current setup is tuned to favor **FraudShield workflow learning** over generic imitation:
- fewer public curriculum rows
- more expert FraudShield rollouts
- lower learning rate
- longer stage-2 adaptation


In [ ]:
%pip uninstall -y unsloth unsloth_zoo trl transformers tokenizers
%pip install -q -U pip

%cd /content
!rm -rf Fraudshield
!git clone https://github.com/DevikaJ2005/Fraudshield.git
%cd /content/Fraudshield
!pip install -q -e ".[rl]"
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"


In [ ]:
import os
from getpass import getpass
from huggingface_hub import login

token = getpass('Enter your HF token (recommended): ')
if token.strip():
    os.environ['HF_TOKEN'] = token.strip()
    login(token=token.strip())
    print('HF login completed.')
else:
    print('No HF token provided.')


In [ ]:
import torch
print('cuda available:', torch.cuda.is_available())
print('device count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('gpu name:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('GPU not available. Enable GPU in Runtime > Change runtime type, then restart.')


In [ ]:
import json
from pathlib import Path

config_path = Path('configs/colab_qlora_grpo.json')
config = json.loads(config_path.read_text(encoding='utf-8'))
print(json.dumps(config, indent=2))


In [ ]:
!python train.py --config configs/colab_qlora_grpo.json

In [ ]:
!python evaluate.py --config configs/colab_qlora_grpo.json

In [ ]:
from IPython.display import Image, display
!find artifacts -name "*.png" -o -name "*.json" | sort

paths = [
    'artifacts/rl_runs/colab_qlora_grpo/loss_vs_steps.png',
    'artifacts/rl_runs/colab_qlora_grpo/reward_vs_steps.png',
    'artifacts/plots/evaluation_rewards.png',
]
for path in paths:
    try:
        display(Image(path))
    except Exception as exc:
        print('Could not display', path, exc)


In [ ]:
import json
from pathlib import Path

summary_candidates = [
    Path('artifacts/rl_runs/colab_qlora_grpo/training_run_summary.json'),
    Path('artifacts/rl_runs/colab_qlora_grpo/evaluation_report.json'),
]
for candidate in summary_candidates:
    if candidate.exists():
        print(f'===== {candidate} =====')
        print(json.dumps(json.loads(candidate.read_text(encoding='utf-8')), indent=2)[:12000])

!zip -r fraudshield_training_outputs.zip artifacts/rl_runs/colab_qlora_grpo artifacts/plots
print('Created fraudshield_training_outputs.zip')
